# 022 — Training: heteroscedastic (beta-weighted NLL)

Retrains the same four heteroscedastic architectures as `021_training_nll.ipynb` — `unet_nll`, `resunet_nll`, `attention_unet_nll`, `efficientnet_unet_nll` — with the **beta-weighted** NLL loss (`scripts.losses.beta_gaussian_nll_loss`, Seitzer et al. 2022) instead of the plain Gaussian NLL, into a **separate checkpoint tree** so `021`'s `gaussian_nll` checkpoints are preserved for comparison (`032_evaluation_v2.ipynb` needs both). Same block layout as `020`/`021` — see `020_training.ipynb`'s title cell for the full table of companion notebooks. `023_training_variants.ipynb` covers `unet_v2`/`unet_restormer`, which used to live in this notebook's Part B before this reorganization split the two unrelated jobs apart.

Only the **artwork-and-mockups** split is used (see §1).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import get_callbacks
from scripts.trainer_nll import compile_model_nll, get_model_nll
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020`/`021` §1 — required so these checkpoints are trained and evaluated under the same conditions as the `gaussian_nll` ones they are compared against.


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss function — beta-weighted NLL

`scripts.losses.beta_gaussian_nll_loss` weights the Gaussian NLL by `stop_gradient(sigma) ** (2 * BETA)`. Plain `gaussian_nll` (`021`) naturally back-propagates a weaker gradient into `mu` wherever `log_var` is high, which can starve high-uncertainty regions of learning signal; beta-weighting counteracts that without changing what is being modeled. `BETA = 0` recovers `gaussian_nll` exactly — see `code-review.md` §7.6 (Seitzer et al. 2022).

As with `021`, `mae`/`ssim`/`psnr` are computed from the `mu` channel only, so they stay comparable across both loss variants and the deterministic architectures.


## 3. Train all four NLL architectures

Same loop as `021_training_nll.ipynb` §3, with `loss_name="beta_nll"` fixed and a **separate checkpoint/log tree** (`models/nll_beta/`, `logs/nll_beta/`) so this run never overwrites `021`'s `gaussian_nll` checkpoints. Set `EPOCHS = 2` for a quick smoke test before committing to a full run.


In [ ]:
ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
LOSS_NAME = "beta_nll"
BETA = 0.5
MODEL_DIR = settings.MODELS_DIR / "nll_beta"
LOG_DIR = settings.LOGS_DIR / "nll_beta"

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}  (loss: {LOSS_NAME}, beta={BETA})")
    print(f"{'=' * 60}")

    model = get_model_nll(arch)
    model = compile_model_nll(
        model, lr=settings.LEARNING_RATE, loss_name=LOSS_NAME, beta=BETA
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}, {LOSS_NAME}): {best_val_loss:.4f}")

## 4. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch} ({LOSS_NAME})")
    plt.show()

## 5. Summary

Checkpoints saved to `models/nll_beta/<arch>/best_model.keras` — the `gaussian_nll` checkpoints from `021_training_nll.ipynb` at `models/nll_gaussian/<arch>/` are untouched. Logs written to `logs/nll_beta/<arch>/`.


In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")